In [ ]:
pip install pysus

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 2.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 3.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 26.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 385.7/385.7 kB 25.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.7/118.7 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 247.7/247.7 kB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.4/78.4 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.5/63.5 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 318.7/318.7 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns

In [8]:
import pandas as pd
from pysus.online_data import SIM
from google.colab import drive
import time

# Montar o Google Drive
drive.mount('/content/drive')

# Lista de todos os estados brasileiros
estados = ['AC', 'AL', 'AP', 'AM', 'BA', 'CE', 'DF', 'ES', 'GO', 'MA',
           'MT', 'MS', 'MG', 'PA', 'PB', 'PR', 'PE', 'PI', 'RJ', 'RN',
           'RS', 'RO', 'RR', 'SC', 'SP', 'SE', 'TO']

# Intervalo de anos solicitado
anos_parte1 = list(range(2000, 2025)) # Ajustado para incluir 2024 se disponível

dados_parte1 = []
total_registros_parte1 = 0

print("INICIANDO COLETA: C50 (Neoplasia Maligna da Mama) - 2000 a 2024")

for ano in anos_parte1:
    print(f"\nProcessando ano {ano}...")
    registros_ano = 0

    for estado in estados:
        print(f"  → Tentando {estado}...")
        try:
            # Baixar dados do estado/ano
            df_estado = SIM.download(groups='CID10', states=estado, years=ano)
            df_estado = df_estado.to_dataframe()

            # FILTRO ALTERADO PARA C50
            df_c50 = df_estado[df_estado['CAUSABAS'].str.startswith('C50', na=False)].copy()

            print(f"  ✓ {estado}: {len(df_estado)} registros totais, {len(df_c50)} C50")

            if len(df_c50) > 0:
                # Adicionar colunas de referência para não perder a origem após o concat
                df_c50['ANO_REF'] = ano
                df_c50['UF_REF'] = estado
                dados_parte1.append(df_c50)
                registros_ano += len(df_c50)

            # Pausa para não sobrecarregar o FTP do DATASUS
            time.sleep(1.5)

        except Exception as e:
            print(f"  ❌ ERRO em {estado}/{ano}: {type(e).__name__}")
            continue

    total_registros_parte1 += registros_ano
    print(f"\nAno {ano}: {registros_ano} registros C50 coletados.")

# Consolidação Final
if dados_parte1:
    df_parte1 = pd.concat(dados_parte1, ignore_index=True)
    # Nome do arquivo atualizado para C50
    caminho = '/content/drive/MyDrive/c50_2000_2024_V0.csv'
    df_parte1.to_csv(caminho, index=False, encoding='utf-8')
    print(f"\n✅ Arquivo salvo no Google Drive: {caminho}")

KeyboardInterrupt: 